16. Ablation GAT và bản cải tiến GAT trên NF-ToN-IoT-v3, GPU (task 5.4, 5.5, GĐ G)

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from pathlib import Path

ALLOW_GPU = True
WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = round(INFO["total_s"] / 3600, 3) if INFO["device"] == "GPU" else 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-9000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


def find(pattern):
    """Đường dẫn đầu tiên khớp glob đệ quy dưới /kaggle/input (báo lỗi rõ nếu không có)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern} trong /kaggle/input (đã gắn đủ nguồn dữ liệu chưa?)"
    return hits[0]


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

import torch
if torch.cuda.is_available():
    INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
INFO["cpu_count"] = os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| máy:", INFO["device"], INFO["gpu"])
assert ALLOW_GPU or INFO["device"] == "CPU", "Chưa được phép dùng GPU: tắt GPU trong cài đặt notebook"

## pytest toàn bộ (dừng nếu lỗi)

In [ ]:
with step("pytest"):
    r = run([sys.executable, "-m", "pytest", "-q"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]

## Ablation GAT một yếu tố mỗi lần và bản cải tiến GAT trên NF-ToN-IoT-v3, GPU
`arch`: GAT cấu hình chính + heads {1, 2, 8} + layers {1, 3} + hidden {8, 32, 64} + không đặc trưng cạnh (10 cấu hình, cũng là log tinh chỉnh GAT).
`improved`: focal loss; residual + LayerNorm; cả hai (task 5.4). 3 seed, chỉ val. Đường học, ngưỡng FPR ≤ 4,08% và checkpoint mỗi lần chạy được lưu.

In [ ]:
DS = "nf_ton_iot_v3"
DATA = os.path.dirname(find(f"{DS}/split.json"))
INFO.update(dataset=DS, data=DATA)
save_info()
with step("gat_grid_arch_improved"):
    run([sys.executable, "-I", "scripts/run_gnn_grid.py", DATA, DS, WORK, "parts=arch,improved"], cwd=REPO)

In [ ]:
import pandas as pd
g = pd.read_csv(f"{WORK}/grid_{DS}.csv")
INFO["grid"] = {"rows": len(g), "configs": g.groupby("part")["cfg_id"].nunique().to_dict(), "seeds": sorted(g["seed"].unique().tolist())}
save_info()
print(json.dumps(INFO["grid"], indent=1))
print(g.groupby("cfg_id")[["f1_macro", "fpr", "recall_attack", "n_params", "train_s"]].mean().round(4).to_string())
print(open(f"{WORK}/run_info.json", encoding="utf-8").read())